# 03 — LangGraph Workflow

This notebook covers LangGraph fundamentals: StateGraph, nodes, edges, conditional routing, human-in-the-loop, and building multi-step agent workflows.

**Kernel:** `ai-engineering`

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"

In [ ]:
from typing import TypedDict, Literal, Annotated
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

model = ChatChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. LangGraph Fundamentals

LangGraph models workflows as a graph:

- **State** — a TypedDict that flows through the graph
- **Nodes** — functions that read/write state
- **Edges** — connections between nodes (fixed or conditional)

```
Input → Node A → Node B → END
           ↓
        (conditional)
           ↓
        Node C → END
```

## 2. Simple Graph — Greeting Router

A minimal example: classify a greeting as formal or casual, then respond accordingly.

In [ ]:
# Define state
class GreetingState(TypedDict):
    user_input: str
    greeting_type: str  # "formal" or "casual"
    response: str

# Define nodes
def classify_greeting(state: GreetingState) -> GreetingState:
    """Classify the greeting as formal or casual."""
    result = model.invoke(f"Classify this greeting as 'formal' or 'casual' (one word only): {state['user_input']}")
    return {"greeting_type": result.content.strip().lower()}

def formal_respond(state: GreetingState) -> GreetingState:
    """Respond with a formal greeting."""
    result = model.invoke(f"Respond to this formal greeting in a professional tone: {state['user_input']}")
    return {"response": result.content}

def casual_respond(state: GreetingState) -> GreetingState:
    """Respond with a casual greeting."""
    result = model.invoke(f"Respond to this casual greeting in a friendly, relaxed tone: {state['user_input']}")
    return {"response": result.content}

# Routing function
def route_greeting(state: GreetingState) -> str:
    if state["greeting_type"] == "formal":
        return "formal"
    return "casual"

# Build the graph
graph = StateGraph(GreetingState)
graph.add_node("classify", classify_greeting)
graph.add_node("formal", formal_respond)
graph.add_node("casual", casual_respond)

graph.set_entry_point("classify")
graph.add_conditional_edges(
    "classify",
    route_greeting,
    {"formal": "formal", "casual": "casual"}
)
graph.add_edge("formal", END)
graph.add_edge("casual", END)

app = graph.compile()

# Visualize the graph structure
print("Nodes:", list(app.get_graph().nodes.keys()))

In [ ]:
# Test with a formal greeting
result = app.invoke({"user_input": "Good afternoon, sir. I hope this message finds you well."})
print(f"Type: {result['greeting_type']}")
print(f"Response: {result['response']}")

In [ ]:
# Test with a casual greeting
result = app.invoke({"user_input": "Hey, what's up!"})
print(f"Type: {result['greeting_type']}")
print(f"Response: {result['response']}")

## 3. Conditional Branching — Customer Support Router

Route customer queries to specialized agents based on intent.

In [ ]:
class SupportState(TypedDict):
    user_message: str
    intent: str
    response: str

def classify_intent(state: SupportState) -> SupportState:
    """Classify user intent as billing, technical, or general."""
    result = model.invoke(
        f"Classify this customer message into exactly one category: 'billing', 'technical', or 'general'. "
        f"Respond with ONLY the category word.\n\nMessage: {state['user_message']}"
    )
    return {"intent": result.content.strip().lower()}

def billing_agent(state: SupportState) -> SupportState:
    """Handle billing inquiries."""
    result = model.invoke(
        f"You are a billing support agent. Respond helpfully and professionally to: {state['user_message']}"
    )
    return {"response": result.content}

def technical_agent(state: SupportState) -> SupportState:
    """Handle technical issues."""
    result = model.invoke(
        f"You are a technical support agent. Provide clear troubleshooting steps for: {state['user_message']}"
    )
    return {"response": result.content}

def general_agent(state: SupportState) -> SupportState:
    """Handle general inquiries."""
    result = model.invoke(
        f"You are a general customer support agent. Respond helpfully to: {state['user_message']}"
    )
    return {"response": result.content}

def route_intent(state: SupportState) -> str:
    intent = state.get("intent", "general")
    if "billing" in intent:
        return "billing"
    elif "technical" in intent:
        return "technical"
    return "general"

# Build graph
support_graph = StateGraph(SupportState)
support_graph.add_node("classify", classify_intent)
support_graph.add_node("billing", billing_agent)
support_graph.add_node("technical", technical_agent)
support_graph.add_node("general", general_agent)

support_graph.set_entry_point("classify")
support_graph.add_conditional_edges(
    "classify",
    route_intent,
    {"billing": "billing", "technical": "technical", "general": "general"}
)
support_graph.add_edge("billing", END)
support_graph.add_edge("technical", END)
support_graph.add_edge("general", END)

support_app = support_graph.compile()

In [ ]:
# Test all three paths
test_messages = [
    "I was charged twice for my subscription this month",
    "The app keeps crashing when I try to upload a file",
    "What are your business hours?"
]

for msg in test_messages:
    result = support_app.invoke({"user_message": msg})
    print(f"Message: {msg}")
    print(f"Intent: {result['intent']}")
    print(f"Response: {result['response'][:100]}...")
    print()

## 4. Human-in-the-Loop

Interrupt execution for human review. Useful when the model's output needs validation before proceeding.

LangGraph uses `interrupt` and checkpointers for this pattern.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt

class ApprovalState(TypedDict):
    task: str
    draft: str
    approved: bool
    final_output: str

def generate_draft(state: ApprovalState) -> ApprovalState:
    """Generate a draft response."""
    result = model.invoke(
        f"Write a brief draft response for this task: {state['task']}\n"
        f"Keep it to 2-3 sentences."
    )
    return {"draft": result.content}

def human_review(state: ApprovalState) -> ApprovalState:
    """Interrupt for human review."""
    # This interrupts the graph and waits for human input
    review = interrupt({
        "draft": state["draft"],
        "message": "Please review this draft. Approve or provide feedback."
    })
    return {"approved": review.get("approved", False)}

def finalize(state: ApprovalState) -> ApprovalState:
    """Finalize the output."""
    if state.get("approved"):
        return {"final_output": state["draft"]}
    else:
        return {"final_output": "[Draft rejected]"}

def should_finalize(state: ApprovalState) -> str:
    if state.get("approved"):
        return "finalize"
    return END

# Build with MemorySaver for checkpointing
checkpointer = MemorySaver()

approval_graph = StateGraph(ApprovalState)
approval_graph.add_node("generate", generate_draft)
approval_graph.add_node("review", human_review)
approval_graph.add_node("finalize", finalize)

approval_graph.set_entry_point("generate")
approval_graph.add_edge("generate", "review")
approval_graph.add_conditional_edges("review", should_finalize, {"finalize": "finalize", END: END})
approval_graph.add_edge("finalize", END)

approval_app = approval_graph.compile(checkpointer=checkpointer)

print("Graph compiled with checkpointing for human-in-the-loop")

In [ ]:
# Run until interruption
config = {"configurable": {"thread_id": "review-1"}}
result = approval_app.invoke(
    {"task": "Write a response declining a refund request politely."},
    config=config
)

print("Graph interrupted for human review.")
print(f"Draft: {result.get('draft')}")

In [ ]:
# Human approves the draft
from langgraph.types import Command

result = approval_app.invoke(
    Command(resume={"approved": True}),
    config=config
)

print(f"Approved: {result.get('approved')}")
print(f"Final output: {result.get('final_output')}")

## 5. Multi-Step Agent Workflow

Build a ReAct-style agent that reasons and acts in a loop.

```
User Input → Agent (reason) → Tool call? → Tool (act) → Agent (reason) → ... → Final Answer
```

In [ ]:
# Define tools for the agent
@tool
def search_knowledge(query: str) -> str:
    """Search for factual information about a topic."""
    # Simulated knowledge base
    knowledge = {
        "python": "Python is a high-level programming language created by Guido van Rossum in 1991.",
        "javascript": "JavaScript is a programming language primarily used for web development.",
        "machine learning": "Machine learning is a subset of AI that enables systems to learn from data.",
        "transformer": "The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need'."
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return value
    return f"No specific information found for: {query}. Try asking about Python, JavaScript, machine learning, or transformers."

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

agent_tools = [search_knowledge, calculate]
model_for_agent = model.bind_tools(agent_tools)
tool_node = ToolNode(agent_tools)

In [ ]:
# Define agent state
class AgentState(TypedDict):
    messages: Annotated[list, "The messages in the conversation"]

# Define nodes
def call_agent(state: AgentState) -> AgentState:
    """Call the LLM to decide the next action."""
    response = model_for_agent.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState) -> str:
    """Decide: use a tool or finish."""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

# Build the agent graph
agent_graph = StateGraph(AgentState)
agent_graph.add_node("agent", call_agent)
agent_graph.add_node("tools", tool_node)

agent_graph.set_entry_point("agent")
agent_graph.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", END: END}
)
agent_graph.add_edge("tools", "agent")  # After tool, go back to agent

agent_app = agent_graph.compile()
print("Agent graph compiled")
print("Nodes:", list(agent_app.get_graph().nodes.keys()))

In [ ]:
# Run the agent
result = agent_app.invoke({
    "messages": [HumanMessage(content="What year was Python created and how old is it (assuming current year 2026)?")]
})

# Print the conversation
for msg in result["messages"]:
    if isinstance(msg, HumanMessage):
        print(f"User: {msg.content}")
    elif isinstance(msg, AIMessage):
        if msg.tool_calls:
            print(f"Agent: Calling tools: {[tc['name'] for tc in msg.tool_calls]}")
        elif msg.content:
            print(f"Agent: {msg.content}")
    elif isinstance(msg, ToolMessage):
        print(f"Tool result: {msg.content}")

In [ ]:
# Test with a multi-step query
result = agent_app.invoke({
    "messages": [HumanMessage(content="Search for information about machine learning, then calculate 10 * 5 + 3.")]
})

for msg in result["messages"]:
    if isinstance(msg, HumanMessage):
        print(f"User: {msg.content}")
    elif isinstance(msg, AIMessage):
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"Agent: Calling {tc['name']}({tc['args']})")
        elif msg.content:
            print(f"Agent: {msg.content}")
    elif isinstance(msg, ToolMessage):
        print(f"Tool result: {msg.content}")

## 6. Graph Visualization

LangGraph can generate a Mermaid diagram of your graph.

In [ ]:
# Generate Mermaid diagram code
print(agent_app.get_graph().draw_mermaid())

## 7. Exercise — Build a Research Agent

Build a graph with these nodes:

1. **`parse_query`** — Extract the topic and task type ("explain" or "calculate") from the user input
2. **`research`** — If task is "explain", search for information
3. **`calculate`** — If task is "calculate", perform the calculation
4. **`summarize`** — Combine all findings into a final response

State schema:
```python
class ResearchState(TypedDict):
    user_input: str
    topic: str
    task_type: str  # "explain" or "calculate"
    findings: list
    final_answer: str
```

Use conditional edges to route based on `task_type`.

In [ ]:
# Your code here
# class ResearchState(TypedDict):
#     ...
#
# def parse_query(state):
#     ...
#
# Build and test the graph

## Key Takeaways

| Concept | What It Does |
|---------|-------------|
| `StateGraph` | Defines the workflow graph with typed state |
| `add_node` | Registers a function as a graph node |
| `add_edge` | Creates a fixed connection between nodes |
| `add_conditional_edges` | Routes to different nodes based on state |
| `set_entry_point` | Sets the starting node |
| `compile()` | Creates a runnable graph |
| `MemorySaver` | Checkpointer for state persistence |
| `interrupt` + `Command(resume=...)` | Human-in-the-loop pattern |

## Next

→ [Exercises](../exercises/exercise-01.md) — Practice building chains, tools, and graphs